# DEAP Learnable Modality-Attention Selection Demo

This notebook is a DEAP-only supplementary demo for the learnable modality-attention selection procedure. It uses all DEAP physiological modality groups and learns softmax-normalized global modality gates through supervised four-class emotion classification.

The goal is to reproduce the paper's learnable attention-based selection idea without manually preset modality weights. The learned weights are initialized uniformly and optimized from data. This notebook is not the main MFMC tri-modal self-supervised learning pipeline.

## Setup

The notebook reads preprocessed `.npy` files from `DEAP/Data_processed` and displays results inline only. It does not save figures, CSV files, checkpoints, or other local outputs.

In [ ]:
from collections import OrderedDict
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path(os.environ.get("TAFFC_MFMC_ROOT", str(Path.cwd().resolve())))
if (PROJECT_ROOT / "MFMC").exists() and not (PROJECT_ROOT / "DEAP").exists():
    PROJECT_ROOT = PROJECT_ROOT / "MFMC"
elif not (PROJECT_ROOT / "DEAP").exists():
    fallback_root = Path("/home/zhengdeyang/TAFFC_MFMC/MFMC")
    if fallback_root.exists():
        PROJECT_ROOT = fallback_root
BASE_DIR = PROJECT_ROOT
DATA_DIR = BASE_DIR / "DEAP" / "Data_processed"

SEED = 13
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 200 if DEVICE.type == "cuda" else 128
EPOCHS = 30
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 8
ENTROPY_REG_WEIGHT = 0.0  # Disabled by default; no sparsity/entropy bias is applied.

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Using device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")

## Load Modalities

The modality order below is the order used by the model and the final attention table. `skt_data.npy` is preferred when present; otherwise the legacy `temp_data.npy` alias is used.

In [ ]:
def load_array(filename, dtype=np.float32):
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run DEAP_Preprocess.py first.")
    return np.load(path).astype(dtype, copy=False)


eeg = load_array("eeg_data.npy")
eog = load_array("eog_data.npy")
emg = load_array("emg_data.npy")
gsr = load_array("gsr_data.npy")
resp = load_array("resp_data.npy")
pleth = load_array("pleth_data.npy")
skt_filename = "skt_data.npy" if (DATA_DIR / "skt_data.npy").exists() else "temp_data.npy"
skt = load_array(skt_filename)

labels = np.load(DATA_DIR / "emotion_labels.npy").astype(np.int64, copy=False)
subjects = np.load(DATA_DIR / "subject.npy").astype(np.int64, copy=False)

modalities = OrderedDict({
    "EEG": eeg,
    "EOG": eog,
    "EMG": emg,
    "GSR": gsr,
    "Respiration": resp,
    "Plethysmograph": pleth,
    "SKT": skt,
})

sample_counts = [array.shape[0] for array in modalities.values()]
assert len(set(sample_counts)) == 1, f"Modality sample counts differ: {sample_counts}"
N = sample_counts[0]
assert len(labels) == N, f"Labels length {len(labels)} != modality samples {N}"
assert len(subjects) == N, f"Subject length {len(subjects)} != modality samples {N}"
assert labels.dtype == np.int64
assert all(array.dtype == np.float32 for array in modalities.values())

shape_table = pd.DataFrame([
    {
        "Modality": name,
        "shape": tuple(array.shape),
        "channels": array.shape[1],
        "time length": array.shape[2],
    }
    for name, array in modalities.items()
])
display(shape_table)
print(f"Labels shape: {labels.shape}; subjects shape: {subjects.shape}; classes: {sorted(np.unique(labels).tolist())}")

## Train/Validation Split

For this supplementary demo, we use one stratified subject-dependent 80/20 split by emotion label. This is intended to expose the attention-selection mechanism, not to claim final paper-level generalization performance.

In [ ]:
indices = np.arange(N)
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=SEED,
    stratify=labels,
)


class DEAPModalityDataset(Dataset):
    def __init__(self, modality_arrays, labels_array, selected_indices):
        self.modality_arrays = modality_arrays
        self.labels_array = labels_array
        self.selected_indices = np.asarray(selected_indices)

    def __len__(self):
        return len(self.selected_indices)

    def __getitem__(self, item):
        idx = self.selected_indices[item]
        x = [torch.from_numpy(array[idx]) for array in self.modality_arrays.values()]
        y = torch.tensor(self.labels_array[idx], dtype=torch.long)
        return x, y


train_dataset = DEAPModalityDataset(modalities, labels, train_idx)
val_dataset = DEAPModalityDataset(modalities, labels, val_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

## Model

Each modality is encoded by a lightweight temporal CNN shared across channels, followed by a cross-channel projection into a 128D modality embedding. The classifier learns one global attention logit per modality. The logits start at zero, so all modalities begin equally weighted, and the final selection reflects what the supervised training process learned.

In [ ]:
class PerModalityEncoder(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.temporal = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=11, padding=5),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(4),
            nn.Conv1d(16, 32, kernel_size=11, padding=5),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(4),
            nn.Conv1d(32, 64, kernel_size=11, padding=5),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(4),
            nn.Conv1d(64, 128, kernel_size=11, padding=5),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.project = nn.Sequential(
            nn.Linear(channels * 128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 128),
        )

    def forward(self, x):
        batch_size, channels, time_length = x.shape
        if channels != self.channels:
            raise ValueError(f"Expected {self.channels} channels, got {channels}")
        x = x.reshape(batch_size * channels, 1, time_length)
        x = self.temporal(x).reshape(batch_size, channels * 128)
        return self.project(x)


class ModalityAttentionClassifier(nn.Module):
    def __init__(self, modality_channels, num_classes=4):
        super().__init__()
        self.modality_names = list(modality_channels.keys())
        self.encoders = nn.ModuleList([
            PerModalityEncoder(channels) for channels in modality_channels.values()
        ])
        self.attn_logits = nn.Parameter(torch.zeros(len(self.modality_names)))
        self.classifier = nn.Sequential(
            nn.Linear(128 * len(self.modality_names), 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def attention_weights(self):
        return torch.softmax(self.attn_logits, dim=0)

    def forward(self, inputs):
        weights = self.attention_weights()
        gated_embeddings = []
        for i, (encoder, x) in enumerate(zip(self.encoders, inputs)):
            embedding = encoder(x)
            gated_embeddings.append(weights[i] * embedding)
        fused = torch.cat(gated_embeddings, dim=1)
        logits = self.classifier(fused)
        return logits, weights


modality_channels = OrderedDict((name, array.shape[1]) for name, array in modalities.items())
model = ModalityAttentionClassifier(modality_channels).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

print(model)
print("Initial attention weights:")
print(pd.Series(
    model.attention_weights().detach().cpu().numpy(),
    index=model.modality_names,
).round(4))

## Training Helpers

In [ ]:
def move_batch_to_device(batch):
    xs, y = batch
    xs = [x.to(DEVICE, non_blocking=True) for x in xs]
    y = y.to(DEVICE, non_blocking=True)
    return xs, y


def entropy_regularization(weights):
    entropy = -(weights * torch.log(weights.clamp_min(1e-8))).sum()
    return ENTROPY_REG_WEIGHT * entropy


def accuracy_from_logits(logits, y):
    preds = logits.argmax(dim=1)
    return (preds == y).sum().item(), y.numel()


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    for batch in loader:
        xs, y = move_batch_to_device(batch)
        optimizer.zero_grad(set_to_none=True)
        logits, weights = model(xs)
        loss = criterion(logits, y) + entropy_regularization(weights)
        loss.backward()
        optimizer.step()

        correct, count = accuracy_from_logits(logits, y)
        total_loss += loss.item() * count
        total_correct += correct
        total_samples += count

    return total_loss / total_samples, total_correct / total_samples


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    for batch in loader:
        xs, y = move_batch_to_device(batch)
        logits, weights = model(xs)
        loss = criterion(logits, y)
        correct, count = accuracy_from_logits(logits, y)
        total_loss += loss.item() * count
        total_correct += correct
        total_samples += count
    return total_loss / total_samples, total_correct / total_samples


def current_attention_dict(model):
    weights = model.attention_weights().detach().cpu().numpy()
    return OrderedDict((name, float(weight)) for name, weight in zip(model.modality_names, weights))


def attention_weights_dataframe(model, modality_channels):
    weights = current_attention_dict(model)
    df = pd.DataFrame({
        "modality": list(weights.keys()),
        "channels": [modality_channels[name] for name in weights.keys()],
        "attention_weight": list(weights.values()),
    }).sort_values("attention_weight", ascending=False, ignore_index=True)
    df.insert(0, "rank", np.arange(1, len(df) + 1))
    df["retained_top3"] = df["rank"] <= 3
    return df

## Train

Early stopping keeps the best model state in memory only. No checkpoint is written to disk.

In [ ]:
history = []
attention_history = []
best_val_acc = -np.inf
best_state = None
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    weights = current_attention_dict(model)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })
    attention_history.append({"epoch": epoch, **weights})

    weight_text = ", ".join(f"{name}={weight:.3f}" for name, weight in weights.items())
    print(
        f"Epoch {epoch:02d} | train_loss={train_loss:.4f} train_acc={train_acc:.3f} "
        f"| val_loss={val_loss:.4f} val_acc={val_acc:.3f} | {weight_text}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        best_state = deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after epoch {epoch}; best epoch was {best_epoch}.")
        break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Loaded best in-memory model from epoch {best_epoch} with val_acc={best_val_acc:.3f}.")

history_df = pd.DataFrame(history)
attention_history_df = pd.DataFrame(attention_history)
display(history_df.tail())

## Final Attention Analysis

The paper-consistent expectation may place EEG, EOG, and SKT among the top modalities, but this notebook does not force that outcome. The retained top-3 set below is whatever the trained classifier learned in this run.

In [ ]:
final_attention_df = attention_weights_dataframe(model, modality_channels)
display(final_attention_df)

retained_modalities = final_attention_df.loc[final_attention_df["retained_top3"], "modality"].tolist()
print("Retained top-3 modalities:", retained_modalities)

## Visualizations

In [ ]:
plot_df = final_attention_df.sort_values("attention_weight", ascending=False)
colors = plot_df["retained_top3"].map({True: "#197278", False: "#C8B6A6"})

plt.figure(figsize=(10, 5))
bars = plt.bar(plot_df["modality"], plot_df["attention_weight"], color=colors)
plt.ylabel("Attention weight")
plt.xlabel("Modality")
plt.title("Learned Global Modality Attention Weights")
plt.ylim(0, max(0.2, float(plot_df["attention_weight"].max()) * 1.2))
plt.xticks(rotation=25, ha="right")
for bar, value in zip(bars, plot_df["attention_weight"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
plt.tight_layout()
plt.show()

In [ ]:
if not history_df.empty:
    plt.figure(figsize=(8, 4))
    plt.plot(history_df["epoch"], history_df["train_acc"], marker="o", label="Train accuracy")
    plt.plot(history_df["epoch"], history_df["val_acc"], marker="o", label="Validation accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training Curves")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

if not attention_history_df.empty:
    plt.figure(figsize=(10, 5))
    for modality in modality_channels.keys():
        plt.plot(attention_history_df["epoch"], attention_history_df[modality], marker="o", label=modality)
    plt.xlabel("Epoch")
    plt.ylabel("Attention weight")
    plt.title("Attention Weight Trajectories")
    plt.legend(ncol=2)
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()